# PipeDream với M = 64 (1F1B theo từng micro-batch) — VGG-16 / Tiny-ImageNet, Kaggle T4 ×2
Kiểm tra cách hiểu thứ hai về baseline của paper (paper_notes §27): PipeDream cập nhật weight sau mỗi 64 mẫu (gấp 3 lần số lần cập nhật mỗi epoch so với M = 192). Yêu cầu: GPU T4 x2, Internet On, dataset `tin-cache`.
Thứ tự trong **một phiên** (~5–6 h): setup → dữ liệu → check → sweep lr (dán bảng cho agent) → bench → *(agent cập nhật lr trong `configs/tin_pipedream_m64.yaml`; chạy `!git pull` ở cell 1 — chỉ đổi YAML nên check vẫn còn hiệu lực)* → run 80 epoch → zip.

## 1. Repo + cài đặt

In [ ]:
import os
REPO = '/kaggle/working/timeprest-reproduction'
if not os.path.exists(REPO):
    !git clone https://github.com/cotda/timeprest-reproduction.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log --oneline -1
!pip install -q -e ".[test]"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpus', torch.cuda.device_count(), '| nccl', torch.cuda.nccl.version())"
!find /kaggle/input -maxdepth 6 \( -name wnids.txt -o -name stats.json \)

## 2. Dữ liệu

In [ ]:
!python -c "from timeprest.config import load_config; from timeprest.data import build_datasets; import json, os; c = load_config('configs/tin_base.yaml'); tr, te = build_datasets(c['data'], 200, 0); print('train', len(tr), 'val(test)', len(te), 'sample', tuple(tr[0][0].shape), '| cache', tr.cache_dir); print(json.load(open(os.path.join(tr.cache_dir, 'stats.json'))))"

## 3. Check D1–D4 cho PipeDream (~3 phút)

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.checks --config configs/tin_quick_pd.yaml

## 4. Sweep lr cho M = 64 (1 GPU, ~20–30 phút) — dán bảng cuối cho agent

In [ ]:
!python -m timeprest.sweep --config configs/tin_pipedream_m64_sweep.yaml

## 5. Bench cùng máy: TiMePReSt, PipeDream M=192, PipeDream M=64 × {cùng máy, 2, 1 Gbit/s} (~25 phút) — dán bảng cuối

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.bench_comm --config configs/tin_bench_m64.yaml

## 6. Run 80 epoch (sau khi agent cập nhật lr và bạn đã `git pull`)

In [ ]:
!cd /kaggle/working/timeprest-reproduction && git pull --ff-only && grep -n 'lr:' configs/tin_pipedream_m64.yaml
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/tin_pipedream_m64.yaml --resume

## 7. Đóng gói

In [ ]:
!cd /kaggle/working && zip -qr /kaggle/working/results_tin_pd_m64.zip runs/tin_vgg16_pipedream_m64/ tin_bench_m64/bench_comm.* tin_bench_m64/*/metrics.csv tin_pipedream_m64_sweep/summary.csv -x '*.pt'
from IPython.display import FileLink
display(FileLink('/kaggle/working/results_tin_pd_m64.zip'))